# 🏗️ Starter — Prediksi Respon Kampanye Marketing (Bab 3)> Notebook eksperimen untuk project starter. **Prasyarat:** semua test hijau> (`python -m unittest discover -s tests -v`). Kalau belum, selesaikan dulu> paket `ml_sederhana/` — notebook ini memakai fungsi-fungsimu.**Aturan main:** semua keputusan (threshold, jumlah klaster, interpretasi)diambil di **validation set**. Test set hanya disentuh di Bagian 6, sekali.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))   # biar `ml_sederhana` bisa diimpor dari notebook

import matplotlib.pyplot as plt
import numpy as np

from ml_sederhana.clustering import kmeans, standardize
from ml_sederhana.data import muat_semua
from ml_sederhana.metrik import precision_recall_f1, roc_auc
from ml_sederhana.model import LogisticRegressionGD

X_train, y_train, X_val, y_val, X_test, y_test, NAMA = muat_semua()
print('train/val/test:', len(y_train), len(y_val), len(y_test),
      '| positif:', y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))


## Bagian 1 — Baseline Dulu, Baru ModelSebelum merayakan accuracy model, hitung dulu baseline bodoh:**prediksi semua pelanggan tidak akan membeli**. Kelas positif cuma ~11%,jadi model bodoh ini otomatis dapat accuracy tinggi.

In [ ]:
# Baseline bodoh: semua prediksi 0
y_bodoh = np.zeros_like(y_val)
acc_bodoh = float((y_bodoh == y_val).mean())
print(f'Accuracy model bodoh di val: {acc_bodoh:.4f}')
print('-> Kalau modelmu cuma unggul sedikit dari ini, ia belum belajar apa-apa.')


In [ ]:
# TODO: latih modelmu (dari ml_sederhana.model), prediksi val (threshold 0.5),
# lalu hitung precision/recall/F1 pakai ml_sederhana.metrik.
model = LogisticRegressionGD(lr=0.1, n_iter=2000).fit(X_train, y_train)

p_val = model.predict_proba(X_val)
pred_val = model.predict(X_val)

# TODO: precision, recall, f1 = precision_recall_f1(...)
# print(f'precision={p:.3f} recall={r:.3f} f1={f1:.3f}')
# -> Bandingkan precision/recall model vs model bodoh (yang nilainya 0 semua).


**Refleksi 1:** accuracy 0.91 vs 0.89 — bedanya tipis. Tapi precision/recallmodel bodoh = 0/0. Metrik mana yang benar-benar menunjukkan model belajar?

## Bagian 2 — Kurva Loss

In [ ]:
# Loss harus monoton turun kalau GD sehat.
plt.figure(figsize=(7, 4))
plt.plot(model.loss_history)
plt.xlabel('iterasi'); plt.ylabel('BCE (train)'); plt.title('Kurva Loss')
plt.grid(alpha=0.3); plt.show()

print(f'loss awal  : {model.loss_history[0]:.4f}  (= ln 2, tebakan 50:50)')
print(f'loss akhir : {model.loss_history[-1]:.4f}')


## Bagian 3 — Eksperimen ThresholdThreshold default 0.5 bukan hukum alam. Geser, catat precision & recall,dan pikirkan **biaya kesalahan** kampanye ini:- **FP** (kirim promo ke yang tak beli) → biaya promo mubazir (murah).- **FN** (lewatkan pelanggan yang mau beli) → kehilangan pendapatan (mahal).

In [ ]:
# TODO: loop threshold 0.05..0.95 (step 0.05). Untuk tiap threshold:
#   - pred = (p_val >= th).astype(int)
#   - hitung precision & recall
#   - simpan ke list
# Lalu print tabel & jawab: kalau FN jauh lebih mahal dari FP,
# threshold berapa yang kamu pilih?
ambang = np.round(np.arange(0.05, 0.96, 0.05), 2)
# for th in ambang: ...


**Refleksi 2:** threshold rendah → recall naik, precision turun. Kampanyeini mau minimal salah melewatkan pembeli. Threshold pilihanmu: ____ karena ____.

## Bagian 4 — ROC-AUC: Menilai Ranking, Bukan KeputusanAUC menjawab: *kalau diambil satu pelanggan positif dan satu negatif acak,seberapa sering model memberi skor lebih tinggi ke yang positif?*AUC tidak terpengaruh threshold — ia menilai urutan, bukan keputusan.

In [ ]:
# TODO: hitung AUC val pakai roc_auc(y_val, p_val), lalu interpretasikan:
#   0.5 = tebak acak, 1.0 = ranking sempurna.
# auc = roc_auc(y_val, p_val)
# print(f'AUC val: {auc:.4f}')

# Sanity check: AUC HARUS sama di threshold mana pun (tidak pakai threshold sama sekali).


## Bagian 5 — Segmen Pelanggan dengan K-Means

In [ ]:
# Gabungkan SEMUA pelanggan (clustering = unsupervised, tidak butuh label)
X_all = np.vstack([X_train, X_val, X_test])
y_all = np.concatenate([y_train, y_val, y_test])

# TODO: standardize -> kmeans(k=2, seed=0) -> hitung ukuran & positive-rate tiap klaster
Z = standardize(X_all)
# labels, centroids, inertia = kmeans(...)
# for k in range(2):
#     anggota = y_all[labels == k]
#     print(f'klaster {k}: {len(anggota)} orang, positive-rate {anggota.mean():.3f}')


**Refleksi 3:** dua klaster punya positive-rate yang berbeda jauh meskik-means TIDAK pernah melihat label. Fitur apa yang paling mungkin memisahkanmereka? (hint: bandingkan centroid dengan NAMA fitur — dan ingat hasil test`test_kontak_ke_fitur_terkuat`.) Kalau budget promo cuma cukup satu segmen,mana yang dipilih?

## Bagian 6 — Sentuh Test Set (SEKALI!)> ⚠ Setelah cell ini dijalankan, eksperimen selesai. Kalau hasil test> mengecewakan dan kamu mengubah model lalu mengetes lagi — itu artinya test> set sudah jadi validation set kedua, dan angkanya jadi terlalu optimis.

In [ ]:
# Pilih threshold TERBAIK dari val (Bagian 3), baru evaluasi sekali di test.
TH_PILIHAN = 0.35   # TODO: ganti dengan pilihanmu dari Bagian 3

p_test = model.predict_proba(X_test)
pred_test = (p_test >= TH_PILIHAN).astype(int)
print(f'accuracy test @th={TH_PILIHAN}: {(pred_test == y_test).mean():.4f}')
print(f'precision/recall test : {precision_recall_f1(y_test, pred_test)}')
print(f'AUC test              : {roc_auc(y_test, p_test):.4f}')


---## 📝 Pertanyaan Analisis (jawab dengan markdown, kata-kata sendiri + angka)1. Kenapa accuracy 0.9125 menipu, dan metrik apa yang memperlihatkan kebenarannya?2. Threshold mana yang kamu pilih untuk kampanye ini? Jelaskan dengan trade-off biaya FP vs FN.3. Kenapa ROC-AUC tidak berubah saat threshold digeser, padahal F1 berubah?4. Justifikasi pemilihan segmen promo dengan angka (positive-rate & ukuran klaster).5. Bandingkan angka val vs test. Seberapa beda? Kapan perbedaan jadi tanda bahaya?Setelah semua terjawab → isi `RUBRIK.md`. Baru bandingkan dengan `solusi.ipynb`.